__Importing Libraries__

In [1]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *


In [2]:
#Creating a SparkSession
spark=SparkSession.builder.appName('Customer Data Analysis').getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/28 17:52:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Reading data from Different Sources 

__Reading CSV file__

In [3]:
%%bash
head -10 ./Data/Superstore.csv

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0,41.9136
2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0,219.582
3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0,6.8714
4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,F

In [4]:
#Read CSV file with headers

df=spark.read.csv("Data/Superstore.csv",header=True)
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [5]:
df.printSchema()

root
 |-- Row ID: string (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: string (nullable = true)



In [6]:
df=spark.read.csv("Data/Superstore.csv",header=True,inferSchema=True,quote='"',escape='"',multiLine=True)
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [7]:
try:
    df.write.parquet("Data/Superstore.parquet")
    print("Parquet formated file Successfully saved")
except :
    print("File is already Saved")

File is already Saved


In [8]:
#Reading Data into parquet format

df=spark.read.parquet("Data/Superstore.parquet")

print("Structure of the Data:")
df.printSchema()

print("--Printing the Data--")
df.show(4)

Structure of the Data:
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)

--Printing the Data--
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-

In [9]:
df = df.withColumnRenamed("Sales", "Price")


date_cols = ["Ship Date", "Order Date"]
for d in date_cols:
    df = df.withColumn(d, to_date(col(d), "M/d/yyyy"))

In [10]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



__Transformation__

In [11]:
selected_column = (df
                    .select('Order ID','Customer ID','Product ID','Price','Quantity','Category')
                    .withColumn("Total_amount", df.Price * df.Quantity)

)
selected_column = selected_column.filter(selected_column.Total_amount > 50000)
selected_column.show()


+--------------+-----------+---------------+--------+--------+---------------+------------------+
|      Order ID|Customer ID|     Product ID|   Price|Quantity|       Category|      Total_amount|
+--------------+-----------+---------------+--------+--------+---------------+------------------+
|CA-2014-139892|   BM-11140|TEC-MA-10000822|8159.952|       8|     Technology|         65279.616|
|CA-2014-145317|   SM-20320|TEC-MA-10002412|22638.48|       6|     Technology|         135830.88|
|US-2016-107440|   BS-11365|TEC-MA-10001047| 9099.93|       7|     Technology|          63699.51|
|CA-2017-129021|   PO-18850|TEC-PH-10001459|4367.896|      13|     Technology|56782.647999999994|
|CA-2016-118689|   TC-20980|TEC-CO-10004722|17499.95|       5|     Technology|          87499.75|
|CA-2017-140151|   RB-19360|TEC-CO-10004722|13999.96|       4|     Technology|          55999.84|
|CA-2016-117121|   AB-10105|OFF-BI-10000545| 9892.74|      13|Office Supplies|         128605.62|
|CA-2015-116638|   J

In [12]:
selected_column=selected_column.groupBy('Category').count()
selected_column.show()

+---------------+-----+
|       Category|count|
+---------------+-----+
|Office Supplies|    1|
|      Furniture|    1|
|     Technology|    6|
+---------------+-----+



__Saving/Writing the Data__

In [13]:
df.write.mode("overwrite").parquet("output")
print("Data saved Successfully")

Data saved Successfully


## Summary

### Pipeline Steps
| Step | Operation | Method |
|---|---|---|
| 1 | Load CSV | `spark.read.csv` with quote/escape handling |
| 2 | Convert to Parquet | `df.write.parquet` — columnar, faster reads |
| 3 | Rename + Cast | `withColumnRenamed`, `cast('double')`, `to_date` |
| 4 | Filter (AND) | `(col('Total_amount') > 50000)` |
| 5 | Derive column | `withColumn('total_amount', Price * quantity)` |
| 6 | Aggregate | `groupBy('Category').agg(count)` |
| 7 | Save output | Parquet  using `write.mode('overwrite')` |

### Key Insights
- **Parquet vs CSV:** Parquet is columnar — queries reading only 2-3 columns skip the rest entirely, reducing I/O significantly on large datasets
- **Lazy Evaluation:** Transformations like `filter()`, `withColumn()` build a DAG but execute only when an action like `.show()` or `.count()` is called
- **Predicate Pushdown:** When reading Parquet, Spark pushes filter conditions into the file reader — only matching row groups are loaded into memory
- **Avoid collect():** On large datasets `collect()` brings all data to the driver — use `.show(n)` instead
- **Wide vs Narrow:** `groupBy()` is a wide transformation causing shuffle — data moves across partitions. `filter()` and `select()` are narrow — no shuffle needed
